In [ ]:
# Script that analyses area log2 fold changes in transcribing and non-transcribing NCs shown in Figure 5

# Imports
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from scipy.stats import ttest_1samp

# Load dataset

df = pd.read_csv('AreasInWindowsNaNMean.csv')

# Identify window columns
windows = [col for col in df.columns if col != 'Signalling']

# Calculate fold change relative to Window 1
df_fc = df.copy()

baseline_window = windows[0]

# Fold change relative to baseline
df_fc[windows] = df_fc[windows].div(df_fc[baseline_window], axis=0)

# Log2 transform
df_log2fc = df_fc.copy()

# Avoid log2(0)
epsilon = 0 #1e-9

df_log2fc[windows] = np.log2(df_log2fc[windows] + epsilon)

# Split signalling vs non-signalling
df_signal = df_log2fc[df_log2fc['Signalling'] == 1].drop(columns=['Signalling'])
df_nosignal = df_log2fc[df_log2fc['Signalling'] == 0].drop(columns=['Signalling'])

# Heatmap plotting
def plot_heatmap(df, title, outfile):

    plt.figure(figsize=(6, 6))

    ax = sns.heatmap(
        df,
        cmap='GnBu',
        center=0,
        vmin=-0.5,
        vmax=1,
        cbar_kws={'label': 'log2 fold change'},
        linewidths=0,
        linecolor='none',
        square=False
    )

    # Rasterize heatmap for cleaner PDFs
    ax.collections[0].set_rasterized(True)

    plt.title(title)
    plt.xlabel("Time window")
    plt.ylabel("Cells")

    plt.tight_layout()

    plt.savefig(
        outfile,
        dpi=300,
        bbox_inches='tight',
        pad_inches=0
    )

    plt.show()

# Plot heatmaps
plot_heatmap(
    df_signal,
    "Log2 fold-change in apical area (Signalling cells)",
    "areas_log2fc_heatmap_signalling.pdf"
)

plot_heatmap(
    df_nosignal,
    "Log2 fold-change in apical area (Non-signalling cells)",
    "areas_log2fc_heatmap_nonsignalling.pdf"
)

# Statistics

def window_stats_and_tests(df_log2fc, label):

    print(f"\n==============================")
    print(f"{label} cells")
    print(f"==============================")

    windows = df_log2fc.columns.tolist()

    # Mean ± SD

    stats = df_log2fc.agg(['mean', 'std']).T

    print("\nMean ± SD log2 fold change:")
    print(stats)

    # One-sample t-tests against baseline (= 0)

    print("\nOne-sample t-tests against baseline (log2FC = 0):")

    for w in windows[1:]:

        vals = df_log2fc[w].dropna()

        tstat, pval = ttest_1samp(vals, popmean=0)

        print(
            f"{w}: "
            f"mean = {vals.mean():.3f}, "
            f"t = {tstat:.3f}, "
            f"p = {pval:.4e}"
        )

    return stats

# Convert to long format

def to_long(df_log2fc):

    df_long = df_log2fc.copy()

    df_long['CellID'] = df_long.index

    df_long = df_long.melt(
        id_vars='CellID',
        var_name='Window',
        value_name='Log2FoldChange'
    )

    return df_long

# Boxplots with paired lines

def plot_boxplot_with_lines(df_long, title, outfile):

    plt.figure(figsize=(7, 5))

    sns.boxplot(
        data=df_long,
        x='Window',
        y='Log2FoldChange',
        color='lightgray',
        showfliers=False
    )

    sns.stripplot(
        data=df_long,
        x='Window',
        y='Log2FoldChange',
        color='black',
        size=4,
        jitter=0.15,
        alpha=0.7
    )

    # Paired lines
    for cell_id, d in df_long.groupby('CellID'):

        plt.plot(
            d['Window'],
            d['Log2FoldChange'],
            color='black',
            alpha=0.3,
            linewidth=0.7
        )

    # Baseline reference
    plt.axhline(
        0,
        color='red',
        linestyle='--',
        linewidth=1
    )

    plt.ylabel("log2 fold change in apical area")
    plt.xlabel("Time window")

    plt.title(title)

    plt.tight_layout()

    plt.savefig(
        outfile,
        dpi=300,
        bbox_inches='tight'
    )

    plt.show()

# Run signalling analysis

stats_signal = window_stats_and_tests(
    df_signal,
    "Signalling"
)

df_signal_long = to_long(df_signal)

plot_boxplot_with_lines(
    df_signal_long,
    "Log2 fold-change in apical area (Signalling cells)",
    "areas_log2fc_boxplot_signalling.pdf"
)

# Run non-signalling analysis

stats_nosignal = window_stats_and_tests(
    df_nosignal,
    "Non-signalling"
)

df_nosignal_long = to_long(df_nosignal)

plot_boxplot_with_lines(
    df_nosignal_long,
    "Log2 fold-change in apical area (Non-signalling cells)",
    "areas_log2fc_boxplot_nonsignalling.pdf"
)

# Save processed dataset

df_log2fc.to_csv(
    "areas_log2fc_dataset_used.csv",
    index=False
)

print("\nSaved processed log2 fold-change dataset.")